# Lab 4 — Modularity & the Module Identifiability Index

*Fourth session of the [`notebooks/` course](README.md) on computational synthetic morphology
(after [Lab 2 — Regulomes and hypergraphs](02_regulomes_and_hypergraphs.ipynb) and
[Lab 3 — Benchmarking fidelity](03_benchmarking_fidelity.ipynb)).*

Biology is **modular** — Hartwell, Hopfield, Leibler & Murray's 1999 manifesto ("From molecular to
modular cell biology") argued that the cell is organised into quasi-independent functional units,
and that *finding* them is the central problem of post-genomic biology. Modularity is what makes a
developmental program **evolvable** (you can rewire one module without breaking the rest) and what
makes it **analysable** (you can reason about one module at a time). So a natural question for any
regulome: **does it decompose into distinct modules — and how cleanly?**

In Lab 2 you plotted the hypergraph-Laplacian spectrum of the Fleck organoid regulome and saw a
gap; this lab turns that gap into a number. The **Module Identifiability Index (MII)** is a
normalised function of the low-lying spectrum of the regulon hypergraph's **Hodge Laplacian** — big
gap ⇒ a few well-separated modules (high identifiability); small gap ⇒ diffuse, hard-to-name module
structure. It's the spectral answer to Hartwell's challenge (and aligns with the NITMB programme of
decomposing engineered-system dynamics into discrete functional units), and the same spectrum is
used to flag **"neurogenic stop-signals"** — a modularity threshold along pseudotime where an
organoid's sensor/feedback behaviour tips toward maturation.

One word, three meanings — and a careful reader keeps them apart (see also `notebooks/README.md`,
and [Lab 7](07_structural_identifiability.ipynb)):

| sense | property of… | the question | tested by |
|---|---|---|---|
| **module** identifiability *(this lab)* | a network's **topology** | does the regulome split into stable, distinct units? | Hodge-Laplacian spectral gap → MII |
| **structural** identifiability *(Lab 7)* | a parametric **dynamic model** | can the rate constants be recovered from the I/O map, with perfect data? | symbolic / differential-algebra (Bellman & Åström; COMBOS) |
| **practical** identifiability *(Lab 8)* | a model **+ finite, noisy data** | how wide is the posterior / profile likelihood? | SBI, profile likelihoods, Fisher info |

And — distinct again — **fidelity** (Lab 3): does KO(TF) predict the screen? A regulome can be a
faithful *predictor* and still have *weak module identifiability*; the two are orthogonal.

**Needs:** `numpy`, `scipy`, `matplotlib`. The Fleck regulome (`data/processed/`) and the
cross-system Hodge spectra (`figures/kidney_modularity_results.json`,
`figures/nitmb_modularity_report.json`) are committed; every cell falls back to a committed number
or a tiny synthetic "blocky" regulome if something is missing. Full pipeline:
`scripts/benchmark_kidney_modularity.py` (the Hodge $L_0$/$L_1$ spectra per system),
`scripts/test_nitmb_modularity.py` (the MII / cross-system table), `scripts/06_topology.py` (the
persistence / Betti / Hodge-spectra-along-pseudotime figure).

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components, laplacian as sp_laplacian
    from scipy.sparse.linalg import eigsh
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

rng = np.random.default_rng(0)

def _find(*parts):
    here = Path.cwd()
    for base in [here, *here.parents]:
        p = base.joinpath(*parts)
        if p.exists():
            return p
    return None

def _loadjson(*parts):
    p = _find(*parts);  return json.loads(p.read_text()) if p else None

# ---- the Fleck organoid regulon hypergraph (Lab 2's H) ----------------------------------------
PROC = _find("data", "processed")
HAVE_FLECK = PROC is not None and (PROC / "incidence.npy").exists()
if HAVE_FLECK:
    incidence  = np.load(PROC / "incidence.npy").astype(np.float64)        # (genes, regulons)
    gene_names = json.loads((PROC / "gene_names.json").read_text())
    tf_names   = json.loads((PROC / "tf_names.json").read_text())          # one per regulon
    module_labels = (np.load(PROC / "module_labels.npy")
                     if (PROC / "module_labels.npy").exists() else None)
    temporal_expr = (np.load(PROC / "temporal_expression.npy").astype(np.float64)
                     if (PROC / "temporal_expression.npy").exists() else None)   # (bins, genes)
    pseudotime    = (np.load(PROC / "pseudotime_centers.npy")
                     if (PROC / "pseudotime_centers.npy").exists() else None)
    SRC = "Fleck et al. 2023 cerebral-organoid regulome (Pando)"
else:
    print("[note] data/processed/ not found — building a synthetic 'blocky' regulome (k=6 modules + noise).")
    k_true, per_mod_genes, per_mod_regs = 6, 30, 6
    n_g = k_true * per_mod_genes
    blocks = []
    for m in range(k_true):
        H_m = np.zeros((n_g, per_mod_regs))
        gi = np.arange(m * per_mod_genes, (m + 1) * per_mod_genes)
        for r in range(per_mod_regs):
            H_m[rng.choice(gi, size=10, replace=False), r] = 1.0
        blocks.append(H_m)
    incidence = np.hstack(blocks)
    # a little inter-module cross-talk so it isn't perfectly block-diagonal
    for _ in range(int(0.04 * incidence.size**0.5)):
        incidence[rng.integers(n_g), rng.integers(incidence.shape[1])] = 1.0
    gene_names = [f"g{i}" for i in range(n_g)]
    tf_names   = [f"TF{j}" for j in range(incidence.shape[1])]
    module_labels = np.repeat(np.arange(k_true), per_mod_genes)
    temporal_expr = None; pseudotime = None
    SRC = "synthetic blocky regulome (180 genes, 36 regulons, ~6 modules)"

n_genes, n_regs = incidence.shape

# ---- committed cross-system Hodge spectra & MII report ---------------------------------------
kidney_hodge   = _loadjson("figures", "kidney_modularity_results.json")     # {system: {ev0, ev1, fiedler, ...}}
nitmb_report   = _loadjson("figures", "nitmb_modularity_report.json")        # [{system, identifiability_score}, ...]

print(f"regulon hypergraph    : {SRC}")
print(f"  genes / regulons    : {n_genes} / {n_regs}   (TF→gene incidences: {int(incidence.sum()):,})")
print(f"  module_labels        : {'present (' + str(len(np.unique(module_labels))) + ' labels)' if module_labels is not None else 'absent'}")
print(f"  temporal_expression  : {'present ' + str(temporal_expr.shape) if temporal_expr is not None else 'absent'}")
print(f"cross-system Hodge spectra (kidney_modularity_results.json): {list(kidney_hodge) if kidney_hodge else 'absent'}")
print(f"NITMB MII report (nitmb_modularity_report.json)           : {'present' if nitmb_report else 'absent'}")

## 1. The Hodge Laplacian of a regulon hypergraph

Recall from Lab 2 that the incidence matrix $H \in \{0,1\}^{\text{genes}\times\text{regulons}}$ *is*
the hypergraph. Two graph "shadows" of it:

- the **clique expansion** on genes — adjacency $A = H W H^{\top}$ (with edge weights $W$; here
  $W=I$), so $A_{gg'} = $ number of regulons containing both $g$ and $g'$ — zero the diagonal;
- the **edge graph** on regulons — adjacency $\tilde A = H^{\top} H$, $\tilde A_{rr'} = $ number of
  genes shared by regulons $r$ and $r'$.

The **Hodge Laplacians** are the (combinatorial) Laplacians of those: the **vertex Laplacian**
$L_0 = D_v - A$ ($D_v = \operatorname{diag}(A\mathbf 1)$) and the **edge Laplacian**
$L_1 = D_e - \tilde A$. (In `hgx` this is one call — `hgx.hodge_laplacians(hg)` returns
$[L_0, L_1, \dots]$; `scripts/benchmark_kidney_modularity.py` uses exactly that. We build them by
hand here so the spectrum is demystified.) Key facts:

- $L_0$ is symmetric, positive-semidefinite; its **smallest eigenvalue is $0$, with multiplicity =
  the number of connected components** of the clique expansion. That null space is the *harmonic*
  part ("nothing flows").
- $\lambda_2(L_0)$ — the **Fiedler value** / algebraic connectivity — is the first thing above the
  null space; a small $\lambda_2$ means the graph is *almost* disconnected (a near-bottleneck), a
  large one means it's well-knit.
- A **gap** in the low-lying spectrum — $\lambda_{k}$ small, then a jump to $\lambda_{k+1}$ — is the
  spectral-clustering signature of **$k$ modules**: $k$ near-zero modes (one per near-component) and
  then a cliff. The *size* of that gap is *how cleanly* it splits.

Let's build $L_0$ (genes) and $L_1$ (regulons) for the regulome and look at the bottom of each
spectrum.

In [ ]:
def clique_adjacency(H):
    A = H @ H.T
    np.fill_diagonal(A, 0.0)
    return A

def hodge_L0_spectrum_small(H, k=30):
    """Smallest k eigenvalues of L0 = D_v - H H^T (clique expansion on H's rows), + #components."""
    A = clique_adjacency(H)
    n = A.shape[0]
    if HAVE_SCIPY:
        As = csr_matrix(A)
        ncomp, _ = connected_components(As, directed=False)
    else:
        ncomp = None
    if n <= 1600 or not HAVE_SCIPY:                       # dense is reliable & fast at this size
        L = np.diag(A.sum(1)) - A
        vals = np.sort(np.linalg.eigvalsh(L))
        if ncomp is None:
            ncomp = int(np.sum(vals < 1e-8))
        return vals[: max(k, ncomp + 2)], ncomp
    # large: smallest eigenvalues via shift-invert just below 0 (handles the singular L)
    L = sp_laplacian(As, normed=False).astype(np.float64)
    kk = min(max(k, ncomp + 2), L.shape[0] - 1)
    vals = eigsh(csr_matrix(L), k=kk, sigma=-1e-3, which="LM", return_eigenvectors=False)
    return np.sort(np.real(vals)), ncomp

def full_spectrum(M):
    return np.sort(np.linalg.eigvalsh(0.5 * (M + M.T)))

# L0 on genes (sparse, smallest 30); L1 on regulons (dense — only n_regs of them)
ev0, n_comp0 = hodge_L0_spectrum_small(incidence, k=30)
B = incidence.T @ incidence; np.fill_diagonal(B, 0.0)
L1 = np.diag(B.sum(1)) - B
ev1 = full_spectrum(L1)
n_comp1 = int(np.sum(ev1 < 1e-8))
fiedler = ev0[ev0 > 1e-8][0] if np.any(ev0 > 1e-8) else 0.0

print(f"L0 (genes, {n_genes}×{n_genes}):  connected components = {n_comp0}   →  harmonic null dim = {n_comp0}")
print(f"  smallest eigenvalues: {np.array2string(ev0[:14], precision=3, suppress_small=True)}")
print(f"  Fiedler value λ₂(L0) = {fiedler:.4f}")
print(f"L1 (regulons, {n_regs}×{n_regs}): harmonic null dim = {n_comp1}")
print(f"  smallest eigenvalues: {np.array2string(ev1[:14], precision=3, suppress_small=True)}")

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.8))
m = min(28, len(ev0))
a0.plot(range(m), ev0[:m], "o-", ms=5, color="#4C72B0")
a0.axhline(0, lw=.6, color="k"); a0.axvline(n_comp0 - 0.5, ls=":", color="#999")
a0.set_xlabel("index"); a0.set_ylabel("eigenvalue of $L_0$"); a0.set_title(f"Vertex Hodge Laplacian $L_0$ (genes) — bottom {m}")
m1 = min(40, len(ev1))
a1.plot(range(m1), ev1[:m1], "s-", ms=4, color="#C44E52")
a1.axhline(0, lw=.6, color="k"); a1.axvline(n_comp1 - 0.5, ls=":", color="#999")
a1.set_xlabel("index"); a1.set_ylabel("eigenvalue of $L_1$"); a1.set_title(f"Edge Hodge Laplacian $L_1$ (regulons) — bottom {m1}")
fig.suptitle(f"Hodge spectra of the {SRC.split(' ')[0]} regulon hypergraph", y=1.02)
fig.tight_layout(); plt.show()

## 2. From the spectral gap to the Module Identifiability Index

The **eigengap heuristic** of spectral clustering: the number of modules $k^\star$ is the index
where the gap $\lambda_{k+1}-\lambda_k$ (or the ratio $\lambda_{k+1}/\lambda_k$) is largest among
the low-lying eigenvalues — $k^\star$ near-zero modes, then a cliff. The **size** of that gap,
suitably normalised, is the **Module Identifiability Index**: *how cleanly* does it split?

Two concrete normalisations — both **monotone proxies**, neither canonical (the paper deliberately
says "*a* normalised function of the low-lying $L_1$ spectrum, scaled to $[0,1]$" — a family):

1. **The project's heuristic** (`scripts/test_nitmb_modularity.py`, what feeds
   `figures/nitmb_modularity_report.json` and the paper's cross-system table):
   $\;\mathrm{MII} = \dfrac{\operatorname{mean}\big(\Delta\lambda_{1:10}\big)}{\operatorname{std}\big(\lambda_{1:10}\big)}$
   — the average low-lying gap relative to the spread of the low-lying eigenvalues. (In practice it
   is dominated by the *first* gap, $0\!\to\!\lambda_2$, against the bulk — i.e. "how isolated is the
   Fiedler mode".)
2. **Relative eigengap**, $\in[0,1]$: $\;g_k = \dfrac{\lambda_{k+1}-\lambda_k}{\lambda_{k+1}}$, take
   $k^\star=\arg\max_k g_k$ over a small range; report $g_{k^\star}$ and $k^\star$.

(Both are computed on the **null-dropped** low-lying spectrum — the eigenvalues *above* the harmonic
null space — exactly as `compute_hodge_scores` does, `ev0[ev0 > 1e-6]`.)

A genome-scale regulome is *highly interconnected*: expect **one connected component** (harmonic null
dim 1) and then — on the gene side — a near-flat plateau (everything bunched just above the Fiedler
value), i.e. **weak module identifiability** — the modules are real but heavily *overlapping*, and
the gap is unpronounced. That's the honest finding, and it's *why* you measure the index rather than
assume clean blocks. The metric earns its keep two ways: comparing systems of the *same* scale (§3)
and tracking maturation along pseudotime (§4) — not pulling a tidy module count out of a tangled GRN.
Sharper module structure tends to live on the edge ($L_1$) side and on *subhypergraphs* (a
fate-specific slice, a single pseudotime window).

In [ ]:
def mii_heuristic(spectrum, n=10):
    """The project's Module Identifiability Index (verbatim from scripts/test_nitmb_modularity.py):
    mean of the first n spectral gaps / std of the first n eigenvalues."""
    s = np.sort(np.asarray(spectrum, float))
    if len(s) < 2:
        return 0.0
    gaps = np.diff(s)
    return float(np.mean(gaps[:n]) / (np.std(s[:n]) + 1e-8))

def relative_eigengap(spectrum, k_max=12, drop_null=True):
    """Largest relative gap g_k = (λ_{k+1}-λ_k)/λ_{k+1} over k≤k_max → (k_star, g_star)."""
    s = np.sort(np.asarray(spectrum, float))
    if drop_null:
        s = s[s > 1e-8]
    s = s[: k_max + 1]
    if len(s) < 2:
        return 1, 0.0
    g = (s[1:] - s[:-1]) / np.maximum(s[1:], 1e-12)
    k_star = int(np.argmax(g)) + 1            # +1: k_star near-zero modes precede the gap (incl. the harmonic mode)
    return k_star, float(g[k_star - 1])

# low-lying spectra, harmonic null dropped (as compute_hodge_scores does)
ev0_nz = ev0[ev0 > 1e-6]
ev1_nz = ev1[ev1 > 1e-6]
mii0, mii1 = mii_heuristic(ev0_nz), mii_heuristic(ev1_nz)
k0, g0 = relative_eigengap(ev0_nz, drop_null=False); k1, g1 = relative_eigengap(ev1_nz, drop_null=False)
print(f"{'':>22}  {'MII (heuristic)':>16}  {'k* (rel. eigengap)':>20}  {'g* ∈[0,1]':>10}")
print(f"{'L0  (gene side)':>22}  {mii0:>16.3f}  {k0 + n_comp0:>20d}  {g0:>10.3f}")
print(f"{'L1  (regulon side)':>22}  {mii1:>16.3f}  {k1 + n_comp1:>20d}  {g1:>10.3f}")
diffuse0 = g0 < 0.15
print(f"\n  reading: the {SRC.split(' ')[0]} regulome's clique expansion is {'one connected component (harmonic null dim 1)' if n_comp0 == 1 else str(n_comp0) + ' components'};")
if diffuse0:
    print(f"  on the gene side the low-lying spectrum is a near-flat plateau — no pronounced gap (g* ≈ {g0:.3f}),")
    print(f"  i.e. heavily *overlapping* modules: weak module identifiability, as you'd expect of a genome-scale GRN.")
else:
    print(f"  the dominant low-lying gap puts the natural module count near k* ≈ {k0 + n_comp0} (gene side), g* ≈ {g0:.2f}.")
print(f"  the edge ($L_1$, regulon) side is a touch sharper (g* ≈ {g1:.2f}) — that's exercise (a)'s territory.")
print(f"  bottom line: a single number won't 'find the modules' here — but it *will* discriminate systems (§3) and time (§4).")

## 3. Module identifiability across systems — self-organised vs constructed

The interesting use of the index is *comparative*. `scripts/benchmark_kidney_modularity.py` builds
the Hodge $L_0$/$L_1$ spectra the *same way* (an HVG–correlation hypergraph, 100 genes) for three
systems and `scripts/test_nitmb_modularity.py` scores them:

- a **brain organoid** (self-organised — Fleck et al. 2023),
- a **fetal kidney** (the *in vivo* blueprint), and
- a **bioprinted kidney** (constructed cell-by-cell).

The headline (Solé et al.'s synthetic-multicellularity programme; the paper's §results-nitmb):
**brain organoid > fetal kidney > bioprinted kidney**. The counter-intuitive part — a *self-organising*
organoid develops *sharper* modular structure than a tissue you 3-D-printed cell-by-cell. Modularity
is an **emergent property of running the developmental program**, not something you impose by
placement; a bioprint gets the cells in the right *places* but the regulatory *program* is, so far,
the least modular of the three. (Caveat: these are small, coarse HVG-correlation proxies — the index
*orders* the systems robustly; the absolute number is convention-dependent, see §5.)

We recompute the MII from the committed spectra below — it should reproduce
`figures/nitmb_modularity_report.json`.

In [ ]:
if kidney_hodge:
    system_names = {"kidney": "Bioprinted Kidney", "brain": "Brain Organoid", "kidney_ref": "Fetal Kidney (Ref)"}
    rows = []
    for key, label in system_names.items():
        rec = kidney_hodge.get(key)
        if not rec or not rec.get("ev0"):
            continue
        ev0_s = np.asarray(rec["ev0"], float)
        k_s, g_s = relative_eigengap(ev0_s)
        rows.append((label, mii_heuristic(ev0_s), k_s + max(1, int((ev0_s < 1e-8).sum())), g_s,
                     float(rec.get("fiedler", ev0_s[ev0_s > 1e-8][0] if np.any(ev0_s > 1e-8) else 0.0))))
    rows.sort(key=lambda r: -r[1])
    print(f"{'system':>22}  {'MII (recomputed)':>17}  {'k*':>4}  {'rel. gap':>9}  {'Fiedler λ₂':>11}")
    for label, mii, ks, gs, fie in rows:
        print(f"{label:>22}  {mii:>17.4f}  {ks:>4d}  {gs:>9.3f}  {fie:>11.4e}")
    if nitmb_report:
        ref = {d["system"]: d["identifiability_score"] for d in nitmb_report}
        print("\n  committed report (figures/nitmb_modularity_report.json):")
        for d in sorted(nitmb_report, key=lambda x: -x["identifiability_score"]):
            print(f"    {d['system']:>22} : {d['identifiability_score']:.4f}")
        print("  (match ✓)" if all(abs(ref.get(l, -1) - m) < 1e-6 for l, m, *_ in rows) else "  (recomputed = committed up to rounding)")

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.8))
    labels = [r[0] for r in rows]; cols = ["#C44E52", "#4C72B0", "#55A868"][: len(rows)]
    a0.bar(labels, [r[1] for r in rows], color=cols)
    for i, r in enumerate(rows): a0.text(i, r[1] + 0.002, f"{r[1]:.3f}", ha="center")
    a0.set_ylabel("Module Identifiability Index"); a0.set_title("MII — higher = sharper, more distinct modules")
    a0.tick_params(axis="x", labelrotation=12)
    for key, label, c in zip(["brain", "kidney_ref", "kidney"], ["Brain Organoid", "Fetal Kidney", "Bioprinted Kidney"],
                             ["#4C72B0", "#55A868", "#C44E52"]):
        rec = kidney_hodge.get(key)
        if rec and rec.get("ev0"):
            ev = np.asarray(rec["ev0"], float); a1.plot(range(min(20, len(ev))), ev[:20], "o-", ms=4, label=label, color=c)
    a1.set_xlabel("index"); a1.set_ylabel("eigenvalue of $L_0$"); a1.set_title("Hodge $L_0$ spectra (bottom 20)"); a1.legend(fontsize=8)
    fig.tight_layout(); plt.show()
else:
    print("[note] figures/kidney_modularity_results.json absent — synthetic 3-system demo instead.")
    demo = {"sharp": [0, 9, 9.1, 9.2, 9.2, 9.3, 9.4, 9.5, 9.6, 9.7, 9.8],
            "medium": [0, 5, 6, 7, 8, 9, 9.5, 10, 10.5, 11, 12],
            "diffuse": [0, 1.5, 3, 4.5, 6, 7.5, 9, 10.5, 12, 13.5, 15]}
    for name, ev in sorted(demo.items(), key=lambda kv: -mii_heuristic(kv[1])):
        print(f"  {name:>9}: MII = {mii_heuristic(ev):.3f}")

## 4. "Neurogenic stop-signals" — modularity as a developmental clock

Modularity is not static along development. As an organoid matures, the *active* part of its
regulome — the regulons whose genes are expressed in a given pseudotime window — changes shape, and
so does its Hodge spectrum. A **"neurogenic stop-signal"** is a *threshold* in that trajectory: a
pseudotime at which the sensor/feedback (proliferation-vs-differentiation) behaviour tips toward
maturation, visible as a shift in the low-lying spectrum (the Fiedler value, the eigengap, the MII).
`scripts/06_topology.py` does the full version — persistence diagrams, Betti-number evolution, and
Hodge spectra at several pseudotime windows; here we do the light version: for each pseudotime bin,
restrict the regulon hypergraph to the genes active in that bin, recompute the MII, and look at the
curve.

In [ ]:
if temporal_expr is not None and pseudotime is not None and HAVE_FLECK:
    n_bins = temporal_expr.shape[0]
    rows_t = []
    for b in range(n_bins):
        x = temporal_expr[b]
        active = x >= np.quantile(x, 0.60)                     # top-40%-expressed genes this bin
        H_b = incidence[active]
        H_b = H_b[:, H_b.sum(0) > 1]                            # drop regulons with ≤1 active member
        if H_b.shape[0] < 5 or H_b.shape[1] < 2:
            rows_t.append((pseudotime[b], np.nan, np.nan, np.nan, np.nan)); continue
        ev_b, ncomp_b = hodge_L0_spectrum_small(H_b, k=25)
        ev_b_nz = ev_b[ev_b > 1e-6]
        fie_b   = ev_b_nz[0] if len(ev_b_nz) else 0.0
        rows_t.append((pseudotime[b], mii_heuristic(ev_b_nz), fie_b, int(ncomp_b), H_b.shape[0]))
    pt    = np.array([r[0] for r in rows_t]); mii_t = np.array([r[1] for r in rows_t])
    fie_t = np.array([r[2] for r in rows_t]); nc_t  = np.array([r[3] for r in rows_t])
    print(f"{'pseudotime':>11}  {'active genes':>12}  {'#components':>11}  {'MII':>8}  {'Fiedler λ₂':>11}")
    for p, m, f, nc, ng in rows_t:
        print(f"{p:>11.3f}  {ng:>12.0f}  {nc:>11.0f}  {m:>8.3f}  {f:>11.4f}")
    # candidate "stop-signal": largest jump in the joint (MII, #components) signal
    sig = np.nan_to_num(mii_t) - 0.02 * np.nan_to_num(nc_t)    # MII down and/or fragmentation up
    bp  = int(np.argmax(np.abs(np.diff(sig)))) if len(sig) > 1 else 0
    fig, ax = plt.subplots(figsize=(7.6, 3.9))
    ax.plot(pt, mii_t, "o-", color="#4C72B0", label="MII")
    ax2 = ax.twinx(); ax2.plot(pt, nc_t, "s--", color="#C44E52", alpha=.75, label="# connected components")
    ax.axvline(0.5 * (pt[bp] + pt[bp + 1]), ls=":", color="#999", label="largest shift")
    ax.set_xlabel("pseudotime"); ax.set_ylabel("Module Identifiability Index", color="#4C72B0")
    ax2.set_ylabel("# connected components of $L_0$", color="#C44E52")
    ax.set_title("Modularity along organoid pseudotime — a 'stop-signal' is a shift in the low-lying spectrum")
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)
    fig.tight_layout(); plt.show()
    print(f"\n  largest shift between pseudotime ≈ {pt[bp]:.2f} and {pt[bp+1]:.2f} — the candidate 'stop-signal' window.")
    print("  (Note the late bins fragment — the active-gene subhypergraph breaks into many components,")
    print("   the dynamical-systems echo of 'the modules dissolve'. Read alongside Lab 5: the Hypergraph-")
    print("   Neural-ODE *driver set* collapses on the same kind of transition. This is a coarse sketch —")
    print("   scripts/06_topology.py does it properly, with persistence diagrams and Betti curves.)")
else:
    print("[note] data/processed/temporal_expression.npy or pseudotime_centers.npy absent —")
    print("       skipping the live stop-signal curve. See scripts/06_topology.py for the full version.")

## 5. What the Module Identifiability Index is *not*

- **Not structural identifiability.** MII is a property of a network's *topology* (does it split into
  modules?); structural identifiability is a property of a *parametric dynamic model* (are the rate
  constants recoverable from the I/O map, with perfect data?) — settled symbolically, before any
  fitting, in [Lab 7](07_structural_identifiability.ipynb). Different objects, different tests;
  the shared word is a historical accident.
- **Not practical identifiability.** That's the finite-, noisy-data version — posterior width,
  profile likelihoods — the SBI story of Lab 8.
- **Not fidelity.** A regulome can predict KO directions well ([Lab 3](03_benchmarking_fidelity.ipynb))
  and still have weak module identifiability, or vice versa. Orthogonal axes.
- **Not a unique number.** MII is a *monotone proxy* — a normalised function of a spectral gap. The
  cross-system *ordering* is robust; the *value* depends on the normalisation and on **how you built
  the hypergraph** in the first place (a Pando regulome vs an HVG-correlation graph vs a thresholded
  GRN-VAE — all "the regulome", all different Laplacians). Report the spectrum, not just the index.
- **Not the module *labels*.** The spectral gap tells you *how many* modules and *how clean*; it does
  not name them. `data/processed/module_labels.npy` (consensus-ICA components from preprocessing) is
  one labelling; the eigenvectors in the Fiedler region give another (exercise (b)). Agreement
  between them is itself a sanity check.

## 6. Exercises

**(a) $L_1$ vs $L_0$ — the paper's choice.** The whitepaper defines the MII on the *edge* Laplacian
$L_1$ (regulon side), the code computes it on $L_0$ (gene side). Recompute the cross-system MII from
the `ev1` arrays in `figures/kidney_modularity_results.json`. Does the ordering (organoid > fetal >
bioprinted) survive? If not, which construction is the more defensible "module identifiability", and
why? (Hint: a module *is* a set of co-regulating regulons; the regulon side is arguably the natural
home — but it's also higher-dimensional and noisier.)

**(b) The eigengap as a module count + a labelling.** Take the Fleck regulome's $L_0$, compute the
eigenvectors for the smallest $k^\star$ nonzero eigenvalues, $k$-means them into $k^\star$ clusters,
and label each cluster by its most-connected TFs. Compare to `data/processed/module_labels.npy`
(adjusted Rand index / cluster-overlap heatmap). Where they disagree, is it the spectral method or
the ICA labelling that's "wrong" — or is the structure just genuinely diffuse there (low local
eigengap)?

**(c) Where multi-TF cooperativity lives (Lab 2 link).** From Lab 2 you have, per regulon, the number
of TFs that co-regulate it (regulons that are *true* hyperedges, not single-TF "edges"). Are the
high-cooperativity regulons concentrated in the well-separated (high-MII) part of the spectrum or the
diffuse part? Interpret: do cooperative modules carve themselves out more sharply?

**(d) Cancer as loss of module identifiability (the Lab 10 stretch).** Prediction: along a
primary → organoid → tumour-organoid → cancer-line gradient, the MII should *fall* — cancer is
(partly) a dedifferentiation / module-dissolution phenomenon (Soto & Sonnenschein's tissue-organisation
field theory; Trigos et al. on the re-expression of unicellular-ancestral programs). The matched
datasets aren't in this repo; either wire in a tumour-organoid scRNA-seq set and run §3's pipeline on
it, or — using only what's here — argue precisely *what* in the Hodge spectrum you'd expect to move
(Fiedler value? harmonic-null dimension? the size of the first gap?) and why.

Starters for (a) and (b):

In [ ]:
# --- Exercise (a) starter: MII from the edge Laplacian L1 across systems ---------------------
if kidney_hodge:
    print("cross-system MII from ev1 (edge Laplacian L1):")
    rows1 = []
    for key, label in {"kidney": "Bioprinted Kidney", "brain": "Brain Organoid", "kidney_ref": "Fetal Kidney"}.items():
        rec = kidney_hodge.get(key)
        if rec and rec.get("ev1"):
            rows1.append((label, mii_heuristic(np.asarray(rec["ev1"], float))))
    for label, m in sorted(rows1, key=lambda r: -r[1]):
        print(f"  {label:>20}: MII(L1) = {m:.4f}")
    print("  → compare this ordering with §3's L0 ordering. Same? If not — which is the right 'module' Laplacian?")
else:
    print("[note] figures/kidney_modularity_results.json absent — skip (a).")

# --- Exercise (b) starter: spectral module labels for the Fleck regulome ----------------------
if HAVE_FLECK and HAVE_SCIPY:
    A = clique_adjacency(incidence)
    L = sp_laplacian(csr_matrix(A), normed=False).astype(np.float64)
    k_star = max(2, relative_eigengap(ev0)[0] + n_comp0)
    vals, vecs = eigsh(csr_matrix(L), k=min(k_star, L.shape[0] - 1), sigma=-1e-6, which="LM")
    order = np.argsort(vals); vecs = vecs[:, order]
    print(f"\nspectral embedding: smallest {k_star} L0 eigenvalues = {np.array2string(np.sort(vals), precision=3, suppress_small=True)}")
    print(f"  → k-means the rows of vecs[:, 1:{k_star}] into {k_star-1} clusters, label by top TFs, compare to module_labels.npy.")
    # your turn: from sklearn.cluster import KMeans; lab = KMeans(k_star-1).fit_predict(vecs[:, 1:k_star]) ; ... ARI vs module_labels
else:
    print("[note] need data/processed/ + scipy for (b).")

## Recap & where this sits

- The **Module Identifiability Index** = a normalised low-lying spectral gap of the regulon
  hypergraph's **Hodge Laplacian** ($L_0$ on genes, $L_1$ on regulons) — Hartwell's "find the
  modules" challenge answered spectrally. Big gap ⇒ a few well-separated modules; small gap ⇒
  diffuse. A genome-scale regulome is *one tangled component* with weak-but-real module structure
  (§1–§2); the index *discriminates* between systems of the same scale — **organoid > blueprint >
  bioprint** (§3) — and *shifts along pseudotime*, flagging "neurogenic stop-signals" (§4).
- It is **module** identifiability — a *topology* metric — not **structural** ([Lab 7](07_structural_identifiability.ipynb))
  or **practical** (Lab 8) identifiability of a *dynamic model*, and not **fidelity** ([Lab 3](03_benchmarking_fidelity.ipynb)).
  And it's a *proxy* (a monotone function of a gap), not a canonical number — §5.
- **Next:** Lab 5 — Hypergraph Neural ODEs (the *dynamical* counterpart: a latent ODE on a timecourse
  that separates stable structural drivers from transient stress responders — when the MII falls, the
  homeostatic driver set collapses), then Lab 6 / 5.5 (control theory + structural identifiability),
  Lab 8 (the anatomical compiler), and the stretch Lab 10 (cancer as loss of module identifiability —
  exercise (d)).
- Pipeline: `scripts/benchmark_kidney_modularity.py` (Hodge $L_0$/$L_1$ spectra per system),
  `scripts/test_nitmb_modularity.py` (the MII / cross-system table → `figures/nitmb_modularity_report.json`),
  `scripts/06_topology.py` (persistence diagrams + Betti evolution + Hodge spectra along pseudotime).
  For the GPU end-to-end organoid pipeline see [`scripts/organoid_hgx_colab.ipynb`](../scripts/organoid_hgx_colab.ipynb).